In [1]:
import json

# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,}
    messages.append(assistant_message)


def chat(messages, system=None, effort="low", stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "output_config": {},
        "stop_sequences": stop_sequences,
    }

    if effort and "haiku" not in model:
        params["output_config"]["effort"] = effort.value

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message

def text_from_message(message):
    return "\n".join( block.text for block in message.content if block.type == "text")

In [17]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
        datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"Setting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "id": {
                            "type": "string",
                            "description": "A short unique identifier for this specific invocation (e.g. 'call_1'), used to match its result back to it when results are returned.",
                        },
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "object",
                            "description": "The arguments to the tool, as a JSON object matching that tool's own input_schema (not an encoded string).",
                        },
                    },
                    "required": ["id", "name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}
pass

In [4]:
from anthropic.types import ToolParam
from datetime import datetime


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format or date_format == "":
        raise ValueError("date_format must not be empty")

    return datetime.now().strftime(date_format)


get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": (
        "Gets the current date and time from the system clock and returns it as a formatted string. "
        "Use this tool whenever the user asks what the current date, time, day of the week, or timestamp is, "
        "or when a task depends on knowing 'now' (e.g., calculating a deadline, checking if something is overdue, "
        "or scheduling a reminder relative to the present moment). The output is a single string formatted "
        "according to the requested date_format using Python's strftime directives; if date_format is omitted, "
        "the tool defaults to '%Y-%m-%d %H:%M:%S' (e.g., '2026-09-09 10:15:17'). Passing an empty string for "
        "date_format raises an error rather than silently falling back to the default."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": (
                    "A Python strftime format string controlling how the current datetime is rendered. "
                    "Examples: '%Y-%m-%d' for just the date (e.g. '2026-09-09'), '%H:%M:%S' for just the time "
                    "(e.g. '10:15:17'), '%A, %B %d, %Y' for a human-readable date (e.g. 'Wednesday, September 09, 2026'). "
                    "Defaults to '%Y-%m-%d %H:%M:%S' when omitted."
                ),
            },
        },
        "required": [],
        "additionalProperties": False,
    },
    "strict": True,
})

In [5]:
# Single tool call
messages = []
add_user_message(messages, "What is the exact time, formatted as %H:%M:%S?")

response = client.messages.create(
    model=model,
    max_tokens=1024,
    messages=messages,
    tools=[get_current_datetime_schema]
)

add_assistant_message(messages, response)

messages

[{'role': 'user', 'content': 'What is the exact time, formatted as %H:%M:%S?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01NWgo5vVBG3Lo7iHeMP6VP3', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]}]

In [6]:
tool_use_result = get_current_datetime(**response.content[0].input)

tool_use_result

'23:24:43'

In [7]:
messages.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": response.content[0].id,
                "content": tool_use_result,
                "is_error": False

            }
        ]
    })

messages

[{'role': 'user', 'content': 'What is the exact time, formatted as %H:%M:%S?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01NWgo5vVBG3Lo7iHeMP6VP3', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01NWgo5vVBG3Lo7iHeMP6VP3',
    'content': '23:24:43',
    'is_error': False}]}]

In [8]:
response = client.messages.create(
    model=model,
    max_tokens=1024,
    messages=messages,
    tools=[get_current_datetime_schema]
)

response

Message(id='msg_011Ceu3X4LMArg4SPngYrU8U', container=None, content=[TextBlock(citations=None, text='The exact time is **23:24:43** (11:24:43 PM in 12-hour format).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=943, output_tokens=29, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [9]:
text_from_message(response)

'The exact time is **23:24:43** (11:24:43 PM in 12-hour format).'

In [14]:
# Multi tool use call support
import json

def run_tool(name: str, id: str ,arguments: dict):
    try:
        fn = available_functions[name]
        result = fn(**arguments)
        return {
            "type": "tool_result",
            "tool_use_id": id,
            "content": json.dumps(result),
            "is_error": False
        }
    except Exception as e:
        return {
            "type": "tool_result",
            "tool_use_id": id,
            "content": f"Error: {e}",
            "is_error": True
        }

def run_tools(message):
    tool_result_blocks = []

    for b in message.content:
        if b.type == "tool_use":
            result = run_tool(name=b.name, id=b.id, arguments=b.input)
            tool_result_blocks.append(result)

    return tool_result_blocks

def run_batch(invocations=[]):
    batch_output = []

    for invocation in invocations:
        result = run_tool(name=invocation["name"], arguments=invocation["arguments"], id=invocation["id"])
        batch_output.append(result)

    return batch_output

available_functions = {
    "add_duration_to_datetime": add_duration_to_datetime,
    "set_reminder": set_reminder,
    "get_current_datetime": get_current_datetime,
    "batch_tool": run_batch
}

In [11]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema, set_reminder_schema, add_duration_to_datetime_schema, batch_tool_schema])
        add_assistant_message(messages, response)

        if(response.stop_reason != "tool_use"):
            print(f"Last message: '{text_from_message(response)}'\n----")
            break

        print(f"Message: '{text_from_message(response)}'\n----")

        tool_use_results = run_tools(response)
        add_user_message(messages, tool_use_results)

    return messages

In [18]:
messages = []

add_user_message(messages, "What is the current time in HH:MM format and in HH:MM SS format?")

run_conversation(messages)

Message: ''
----
Last message: 'The current time is:
- **HH:MM format**: 23:37
- **HH:MM:SS format**: 23:37:58'
----


[{'role': 'user',
  'content': 'What is the current time in HH:MM format and in HH:MM SS format?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01TG2jdmwiUapNnWVDQ6Pyvp', caller=DirectCaller(type='direct'), input={'invocations': [{'id': 'time_hhmm', 'name': 'get_current_datetime', 'arguments': {'date_format': '%H:%M'}}, {'id': 'time_hhmmss', 'name': 'get_current_datetime', 'arguments': {'date_format': '%H:%M:%S'}}]}, name='batch_tool', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01TG2jdmwiUapNnWVDQ6Pyvp',
    'content': '[{"type": "tool_result", "tool_use_id": "time_hhmm", "content": "\\"23:37\\"", "is_error": false}, {"type": "tool_result", "tool_use_id": "time_hhmmss", "content": "\\"23:37:58\\"", "is_error": false}]',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The current time is:\n- **HH:MM format**: 23:37\n- **HH:MM:SS format**: 23:37:58'

In [19]:
messages = []

add_user_message(messages, "Set a reminder for my doctor appointment. It's 177 days after January 1st 2027")

run_conversation(messages)

Message: 'I'll help you set a reminder for your doctor appointment. First, let me calculate the exact date that is 177 days after January 1st, 2027.'
----
Message: 'Perfect! Your doctor appointment is on Sunday, June 27, 2027. Now I'll set a reminder for that date. I'll set it for 9:00 AM on that day to give you a good morning reminder:'
----
Setting the following reminder for 2027-06-27T09:00:00:
Doctor appointment
----
Last message: 'Done! I've set a reminder for your doctor appointment on **Sunday, June 27, 2027 at 9:00 AM**. You'll receive a notification at that time.'
----


[{'role': 'user',
  'content': "Set a reminder for my doctor appointment. It's 177 days after January 1st 2027"},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you set a reminder for your doctor appointment. First, let me calculate the exact date that is 177 days after January 1st, 2027.", type='text'),
   ToolUseBlock(id='toolu_01J4s9WQMfqwdHMf11YGiWoy', caller=DirectCaller(type='direct'), input={'datetime_str': '2027-01-01', 'duration': 177, 'unit': 'days'}, name='add_duration_to_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01J4s9WQMfqwdHMf11YGiWoy',
    'content': '"Sunday, June 27, 2027 12:00:00 AM"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Perfect! Your doctor appointment is on Sunday, June 27, 2027. Now I'll set a reminder for that date. I'll set it for 9:00 AM on that day to give you a good morning reminder

In [20]:
messages = []

add_user_message(messages, """
Set 2 reminders for January 1st 2027 at 8AM
- I have a dental appointment
- Plan tax return
""")

run_conversation(messages)

Message: ''
----
Setting the following reminder for 2027-01-01T08:00:00:
I have a dental appointment
----
Setting the following reminder for 2027-01-01T08:00:00:
Plan tax return
----
Last message: 'Perfect! I've set both reminders for January 1st, 2027 at 8:00 AM:
1. ✓ "I have a dental appointment"
2. ✓ "Plan tax return"

You'll receive notifications at that time for both reminders.'
----


[{'role': 'user',
  'content': '\nSet 2 reminders for January 1st 2027 at 8AM\n- I have a dental appointment\n- Plan tax return\n'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01TExBJbDFbQg47dyTSueatt', caller=DirectCaller(type='direct'), input={'invocations': [{'id': 'reminder_1', 'name': 'set_reminder', 'arguments': {'content': 'I have a dental appointment', 'timestamp': '2027-01-01T08:00:00'}}, {'id': 'reminder_2', 'name': 'set_reminder', 'arguments': {'content': 'Plan tax return', 'timestamp': '2027-01-01T08:00:00'}}]}, name='batch_tool', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01TExBJbDFbQg47dyTSueatt',
    'content': '[{"type": "tool_result", "tool_use_id": "reminder_1", "content": "null", "is_error": false}, {"type": "tool_result", "tool_use_id": "reminder_2", "content": "null", "is_error": false}]',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=